In [2]:
import sys
print(sys.executable)

c:\Users\prith\anaconda3\python.exe


In [1]:
!pip install streamlit


In [1]:
import os

# --- CRASH FIX FOR WINDOWS/ANACONDA ---
# This stops duplicate OpenMP runtime libraries from crashing the notebook kernel
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Define paths
TRAIN_PATH = "Data set/Train"
TEST_PATH = "Data set/Test"

verify_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor()
])

print("Checking environment...")
print(f"PyTorch version successfully loaded: {torch.__version__}")

try:
    # Load datasets
    train_dataset = ImageFolder(root=TRAIN_PATH, transform=verify_transform)
    test_dataset = ImageFolder(root=TEST_PATH, transform=verify_transform)
    
    print("\n--- Dataset Setup Successful! ---")
    print(f"Detected Classes/Categories: {train_dataset.classes}")
    print(f"Number of Training images found: {len(train_dataset)}")
    print(f"Number of Testing images found: {len(test_dataset)}")

except Exception as e:
    print("\n--- Dataset Loading Error ---")
    print(e)
    print("\nTip: Make sure you have actually created category folders inside Train and Test folders!")

Checking environment...
PyTorch version successfully loaded: 2.12.0+cpu

--- Dataset Setup Successful! ---
Detected Classes/Categories: ['cars', 'cats', 'dogs', 'flowers']
Number of Training images found: 320
Number of Testing images found: 80


In [2]:
import torch.nn as nn
import torch.nn.functional as F

# 1. Define the Data Loaders to break our dataset into small batches
# Batch size means the model looks at 16 images at a time while training
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 2. Build the CNN Class
class ImageClassifierCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ImageClassifierCNN, self).__init__()
        
        # Convolution Layer 1: Takes 3 color channels (RGB). Outputs 12 features.
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=12, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(num_features=12) # Normalizes layer outputs
        
        # Max Pooling: Cuts image dimensions in half (e.g., 150x150 becomes 75x75)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Convolution Layer 2: Takes 12 features, outputs 24 features.
        self.conv2 = nn.Conv2d(in_channels=12, out_channels=24, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(num_features=24)
        
        # Convolution Layer 3: Takes 24 features, outputs 32 features.
        self.conv3 = nn.Conv2d(in_channels=24, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(num_features=32)
        
        # After passing through 3 MaxPool layers, a 150x150 image shrinks down to 18x18 pixels
        # 32 output features * 18 * 18 = 10,368 flattened nodes
        self.fc = nn.Linear(in_features=32 * 18 * 18, out_features=num_classes)
        
    def forward(self, x):
        # Layer 1: Conv -> Batch Normalization -> ReLU Activation -> MaxPool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        
        # Layer 2: Conv -> Batch Normalization -> ReLU Activation -> MaxPool
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        
        # Layer 3: Conv -> Batch Normalization -> ReLU Activation -> MaxPool
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        # Flatten the multi-dimensional feature map into a 1D vector
        x = x.view(-1, 32 * 18 * 18)
        
        # Final fully connected layer to output class predictions
        x = self.fc(x)
        return x

# 3. Instantiate the model and move it to CPU (or GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImageClassifierCNN(num_classes=4).to(device)

print("--- CNN Architecture Built Successfully ---")
print(model)

--- CNN Architecture Built Successfully ---
ImageClassifierCNN(
  (conv1): Conv2d(3, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(12, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(12, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(24, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (fc): Linear(in_features=10368, out_features=4, bias=True)
)


In [3]:
from torch.optim import Adam

# 1. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001) # Learning rate controls step size

# 2. Configuration Parameters
num_epochs = 10
best_accuracy = 0.0

print("Starting Training Loop...")
print("==================================================")

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Reset the calculated gradients to zero before backprop
        optimizer.zero_grad()
        
        # Forward pass: compute predictions
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass: compute gradients and update weights
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data).item()
        
    epoch_loss = running_loss / len(train_dataset)
    epoch_train_acc = correct_train / len(train_dataset)
    
    # --- TESTING/VALIDATION PHASE ---
    model.eval() # Set model to evaluation mode (turns off dropout/batchnorm updates)
    correct_test = 0
    
    with torch.no_grad(): # Turning off gradient tracking saves memory and computation
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct_test += torch.sum(preds == labels.data).item()
            
    epoch_test_acc = correct_test / len(test_dataset)
    
    print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | Test Acc: {epoch_test_acc*100:.2f}%")
    
    # Save our absolute best model configuration weights
    if epoch_test_acc > best_accuracy:
        torch.save(model.state_dict(), 'best_image_classifier.pth')
        best_accuracy = epoch_test_acc
        print("--> Saved new best checkpoint weights!")
        
print("==================================================")
print(f"Training Complete! Best Test Accuracy Achieved: {best_accuracy*100:.2f}%")

Starting Training Loop...
Epoch 01/10 | Train Loss: 0.2506 | Train Acc: 91.88% | Test Acc: 60.00%
--> Saved new best checkpoint weights!
Epoch 02/10 | Train Loss: 0.0015 | Train Acc: 100.00% | Test Acc: 97.50%
--> Saved new best checkpoint weights!
Epoch 03/10 | Train Loss: 0.0004 | Train Acc: 100.00% | Test Acc: 100.00%
--> Saved new best checkpoint weights!
Epoch 04/10 | Train Loss: 0.0002 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch 05/10 | Train Loss: 0.0001 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch 06/10 | Train Loss: 0.0001 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch 07/10 | Train Loss: 0.0001 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch 08/10 | Train Loss: 0.0000 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch 09/10 | Train Loss: 0.0001 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch 10/10 | Train Loss: 0.0000 | Train Acc: 100.00% | Test Acc: 100.00%
Training Complete! Best Test Accuracy Achieved: 100.00%


In [4]:
import torch
import torchvision.transforms as transforms
from PIL import Image

# 1. Re-initialize an empty model framework and load our trained weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
trained_model = ImageClassifierCNN(num_classes=4)
trained_model.load_state_dict(torch.load('best_image_classifier.pth'))
trained_model.to(device)
trained_model.eval() # Put the model into evaluation mode

# 2. Define standard inference transform (must match your training image shape)
inference_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    # Normalizing with standard ImageNet or basic scale transforms used earlier
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) 
])

# 3. Create the prediction pipeline function
def classify_new_image(image_path):
    try:
        # Load the image file
        img = Image.open(image_path).convert('RGB')
        
        # Apply identical transformations and add mini-batch dimension via unsqueeze
        img_tensor = inference_transform(img).float()
        img_tensor = img_tensor.unsqueeze(0).to(device) 
        
        # Pass the processed tensor forward through the network
        with torch.no_grad():
            output = trained_model(img_tensor)
            
        # Extract the array index with the highest probability
        _, predicted_idx = torch.max(output.data, 1)
        
        # Map the numeric index back to your class text labels
        class_names = ['cars', 'cats', 'dogs', 'flowers']
        result_label = class_names[predicted_idx.item()]
        
        print(f"Prediction Result: This image contains a -> **{result_label.upper()}**")
        
    except Exception as e:
        print(f"Error reading image: {e}")

# ==========================================
# TEST IT OUT: Put any random test image path here!
# ==========================================
classify_new_image("C:/Users/prith/OneDrive/Desktop/Trash/Testing image.png")

Prediction Result: This image contains a -> **CARS**
